In [1]:
import pandas as pd

df = pd.read_csv("Twitter_Data.csv")

print(df.head())
print(df.info())
print(df.columns)

                                          clean_text  category
0  when modi promised “minimum government maximum...      -1.0
1  talk all the nonsense and continue all the dra...       0.0
2  what did just say vote for modi  welcome bjp t...       1.0
3  asking his supporters prefix chowkidar their n...       1.0
4  answer who among these the most powerful world...       1.0
<class 'pandas.DataFrame'>
RangeIndex: 162980 entries, 0 to 162979
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   clean_text  162976 non-null  str    
 1   category    162973 non-null  float64
dtypes: float64(1), str(1)
memory usage: 2.5 MB
None
Index(['clean_text', 'category'], dtype='str')


In [2]:
print(df['category'].value_counts())

category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64


In [3]:
df.rename(columns={'category': 'sentiment'}, inplace=True)

In [4]:
print(df.columns)
print(df['sentiment'].value_counts())

Index(['clean_text', 'sentiment'], dtype='str')
sentiment
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64


In [5]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove special characters & numbers
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    # Join back
    return " ".join(words)

[nltk_data] Downloading package stopwords to C:\Users\HP 745
[nltk_data]     G6\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
df.dropna(inplace=True)
df['processed_text'] = df['clean_text'].apply(preprocess_text)

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

In [9]:
y = df['sentiment']

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [11]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [12]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [13]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

In [15]:
print("Logistic Regression")
evaluate(y_test, y_pred_lr)

print("\nNaive Bayes")
evaluate(y_test, y_pred_nb)

print("\nDecision Tree")
evaluate(y_test, y_pred_dt)

Logistic Regression
Accuracy: 0.9267043014051666
Precision: 0.9265995185208352
Recall: 0.9267043014051666
F1 Score: 0.9258915466919209

Naive Bayes
Accuracy: 0.7374363379763147
Precision: 0.7860468077001568
Recall: 0.7374363379763147
F1 Score: 0.7246016153037766

Decision Tree
Accuracy: 0.8544824200773149
Precision: 0.8534022989829286
Recall: 0.8544824200773149
F1 Score: 0.8538833947715735
